# Test-suite to see how our custom cho_solve function behaves with different input shapes and broadcasting scenarios.

---
## Setup

In [1]:
# Jax configuration
USE_JIT = True
USE_X64 = False
DEBUG_NANS = False
VERBOSE = False

In [2]:
# Standard library imports
import os
os.environ['JAX_ENABLE_X64'] = str(USE_X64).lower()

import logging
logging.basicConfig(level=logging.INFO, format='%(asctime)s - %(levelname)s - %(message)s')

In [18]:
# Third party
import jax
jax.config.update("jax_disable_jit", not USE_JIT)
jax.config.update("jax_debug_nans", DEBUG_NANS)

from jax import jit
import jax.numpy as jnp
import jax.scipy as jsp

import numpy as np

from kernax import BatchKernel

In [4]:
# Local imports
from MagmaClustPy.custom_kernels import RBFKernel
from MagmaClustPy.linalg import cho_factor

INFO:2025-11-25 15:46:15,190:jax._src.xla_bridge:812: Unable to initialize backend 'tpu': INTERNAL: Failed to open libtpu.so: dlopen(libtpu.so, 0x0001): tried: 'libtpu.so' (no such file), '/System/Volumes/Preboot/Cryptexes/OSlibtpu.so' (no such file), '/opt/miniconda3/envs/MagmaClustPy/bin/../lib/libtpu.so' (no such file), '/usr/lib/libtpu.so' (no such file, not in dyld cache), 'libtpu.so' (no such file)
2025-11-25 15:46:15,190 - INFO - Unable to initialize backend 'tpu': INTERNAL: Failed to open libtpu.so: dlopen(libtpu.so, 0x0001): tried: 'libtpu.so' (no such file), '/System/Volumes/Preboot/Cryptexes/OSlibtpu.so' (no such file), '/opt/miniconda3/envs/MagmaClustPy/bin/../lib/libtpu.so' (no such file), '/usr/lib/libtpu.so' (no such file, not in dyld cache), 'libtpu.so' (no such file)


In [5]:
# Config
key = jax.random.PRNGKey(42)

---
## Data

---
## Sandbox

In [6]:
kern = RBFKernel(length_scale=jnp.array(0.3), variance=jnp.array(1.))

In [7]:
def cho_solve(factor: jnp.ndarray, result: jnp.ndarray) -> jnp.ndarray:
	"""
	Wrapper around jax.scipy.linalg.cho_solve to solve the linear system factor @ X = result, as we always use the upper version
	of Cholesky factorisation in the whole codebase.

	:param factor: The Cholesky factorisation of the covariance matrix A (output of cho_factor).
	:param result: The right-hand side matrix or vector to solve for.

	:return: The solution X such that factor @ X = result.

	Special notes on broadcasting:
	- if factor is a matrix (shape M, M) and result is a vector (shape M,), the output will be a vector (shape M,).
	- if factor is a matrix (shape M, M) and result is a matrix (shape M, N), the output will be a matrix (shape M, N).
	- if factor is a matrix (shape M, M) and result is a batch of matrices (shape B, M, N), the output will be a batch of matrices (shape B, M, N).
	- if factor is a batch of matrices (shape B, M, M) and result is a vector (shape M,), the output will be a batch of vectors (shape B, M).
	- if factor is a batch of matrices (shape B, M, M) and result is a matrix (shape M, N), the output will be a batch of matrices (shape B, M, N).
	- if factor is a batch of matrices (shape B, M, M) and result is a batch of vectors (shape B, M), the output will be a batch of vectors (shape B, M).
	- if factor is a batch of matrices (shape B, M, M) and result is a batch of matrices (shape B, M, N), the output will be a batch of matrices (shape B, M, N).

	The broadcasting has one ambiguous case: when factor.ndim==3 and result.ndim==2. Depending on the first dimension, we have two resolutions:
	- (B, M, M) + (M, M) -> (B, M, M), aka each matrix in the batch is solved against the same matrix
	- (B, M, M) + (B, M) -> (B, M), aka each matrix in the batch is solved against its corresponding vector
	If B == M, we have no way of knowing which option to choose. By default, the first option is picked.
	If you want to avoid that, you can add a dimension at the end of the vectors batch and squeeze the result, e.g: (B, M, M) + (B, M, 1) -> (B, M, 1)
	"""
	# Handle different broadcasting scenarios
	if factor.ndim == 2:
		# factor is (M, M)
		if result.ndim == 2 and result.shape[1] == factor.shape[0] and result.shape[0] != factor.shape[0]:
			# Case: (M, M) + (B, M) -> (B, M): transpose result to (M, B), solve, then transpose back
			return jsp.linalg.cho_solve((factor, False), result.T).T
		else:
			# Standard cases: (M, M) + (M,) or (M, M) + (M, N)
			return jsp.linalg.cho_solve((factor, False), result)
	elif factor.ndim == 3:
		# factor is (B, M, M)
		if result.ndim == 1:
			# (B, M, M) + (M,) -> (B, M): broadcast result to match batch dimension
			result = jnp.broadcast_to(result, (factor.shape[0], result.shape[0]))
		elif result.ndim == 2:
			# Handle ambiguous case: (B, M, M) + (B, M) vs (B, M, M) + (M, N)
			if result.shape[0] == factor.shape[0] and result.shape[1] == factor.shape[1]:
				# Case (B, M, M) + (B, M) -> (B, M): already compatible
				pass
			elif result.shape[0] == factor.shape[1]:
				# Case (B, M, M) + (M, N) -> (B, M, N): broadcast to batch dimension
				result = jnp.broadcast_to(result, (factor.shape[0],) + result.shape)
		# For (B, M, M) + (B, M, N), no broadcasting needed
	
	return jsp.linalg.cho_solve((factor, False), result)

### Shape (N, N) + (N,) -> (N,)

In [8]:
key, subkey1, subkey2 = jax.random.split(key, 3)
inputs = jax.random.normal(subkey1, (5, 1))
outputs = jax.random.normal(subkey2, (5,))
cov = kern(inputs)  # Shape (5, 5)
factor = cho_factor(cov)
factor.shape, outputs.shape

((5, 5), (5,))

In [9]:
jsp.linalg.cho_solve(jsp.linalg.cho_factor(cov), outputs).shape

(5,)

In [10]:
cho_solve(factor, outputs).shape

(5,)

### Shape (N, N) + (N, N) -> (N, N) or (N, N) + (B, N) -> (B, N)

In [11]:
key, subkey1, subkey2 = jax.random.split(key, 3)
inputs = jax.random.normal(subkey1, (5, 1))
outputs = jax.random.normal(subkey2, (10, 5))
cov = kern(inputs)  # Shape (5, 5)
factor = cho_factor(cov)
factor.shape, outputs.shape

((5, 5), (10, 5))

In [12]:
reshaped_outputs = outputs.T
reshaped_outputs.shape

(5, 10)

In [13]:
jsp.linalg.cho_solve(jsp.linalg.cho_factor(cov), reshaped_outputs).T.shape

(10, 5)

In [14]:
cho_solve(factor, outputs).shape

(10, 5)

In [15]:
# Check that reshaping doesn't modify results
jnp.allclose(jsp.linalg.cho_solve(jsp.linalg.cho_factor(cov), outputs[0]), jsp.linalg.cho_solve(jsp.linalg.cho_factor(cov), reshaped_outputs).T[0])

Array(True, dtype=bool)

In [16]:
# Check that the all-in one function works accordingly
jnp.allclose(jsp.linalg.cho_solve(jsp.linalg.cho_factor(cov), outputs[0]), cho_solve(factor, outputs)[0])

Array(True, dtype=bool)

### Shape (B, N, N) + (N,) -> (B, N)

In [19]:
key, subkey1, subkey2 = jax.random.split(key, 3)
inputs = jax.random.normal(subkey1, (10, 5, 1))
outputs = jax.random.normal(subkey2, (5,))
cov = BatchKernel(kern, batch_size=inputs.shape[0])(inputs)  # Shape (10, 5, 5)
factor = cho_factor(cov)
factor.shape, outputs.shape

((10, 5, 5), (5,))

In [20]:
reshaped_outputs = jnp.broadcast_to(outputs, (factor.shape[0],)+outputs.shape)
reshaped_outputs.shape

(10, 5)

In [21]:
jsp.linalg.cho_solve(jsp.linalg.cho_factor(cov), reshaped_outputs).shape

(10, 5)

In [22]:
cho_solve(factor, outputs).shape

(10, 5)

In [23]:
# Check that reshaping doesn't modify results
jnp.allclose(jsp.linalg.cho_solve(jsp.linalg.cho_factor(cov[0]), outputs), jsp.linalg.cho_solve(jsp.linalg.cho_factor(cov), reshaped_outputs)[0])

Array(True, dtype=bool)

In [24]:
# Check that the all-in one function works accordingly
jnp.allclose(cho_solve(factor, outputs)[0], jsp.linalg.cho_solve((jsp.linalg.cho_factor(cov)[0][0], False), outputs))

Array(True, dtype=bool)

### Shape (B, N, N) + (B, N) -> (B, N)

In [25]:
key, subkey1, subkey2 = jax.random.split(key, 3)
inputs = jax.random.normal(subkey1, (10, 5, 1))
outputs = jax.random.normal(subkey2, (10, 5))
cov = BatchKernel(kern, batch_size=inputs.shape[0])(inputs)  # Shape (10, 5, 5)
factor = cho_factor(cov)
factor.shape, outputs.shape

((10, 5, 5), (10, 5))

In [26]:
reshaped_outputs = outputs
reshaped_outputs.shape

(10, 5)

In [27]:
jsp.linalg.cho_solve(jsp.linalg.cho_factor(cov), reshaped_outputs).shape

(10, 5)

In [28]:
cho_solve(factor, outputs).shape

(10, 5)

In [29]:
# Check that reshaping doesn't modify results
jnp.allclose(jsp.linalg.cho_solve(jsp.linalg.cho_factor(cov[0]), outputs[0]), jsp.linalg.cho_solve(jsp.linalg.cho_factor(cov), reshaped_outputs)[0])

Array(True, dtype=bool)

In [30]:
# Check that the all-in one function works accordingly
jnp.allclose(cho_solve(factor, outputs)[0], jsp.linalg.cho_solve((jsp.linalg.cho_factor(cov)[0][0], False), outputs[0]))

Array(True, dtype=bool)

### Shape (B, N, N) + (N, N) -> (B, N, N)

In [31]:
key, subkey1, subkey2 = jax.random.split(key, 3)
inputs = jax.random.normal(subkey1, (10, 5, 1))
outputs = jax.random.normal(subkey2, (5, 5))
cov = BatchKernel(kern, batch_size=inputs.shape[0])(inputs)  # Shape (10, 5, 5)
factor = cho_factor(cov)
factor.shape, outputs.shape

((10, 5, 5), (5, 5))

In [32]:
reshaped_outputs = jnp.broadcast_to(outputs, (factor.shape[0],)+outputs.shape)
reshaped_outputs.shape

(10, 5, 5)

In [33]:
jsp.linalg.cho_solve(jsp.linalg.cho_factor(cov), reshaped_outputs).shape

(10, 5, 5)

In [34]:
cho_solve(factor, outputs).shape

(10, 5, 5)

In [35]:
# Check that reshaping doesn't modify results
jnp.allclose(jsp.linalg.cho_solve(jsp.linalg.cho_factor(cov[0]), outputs), jsp.linalg.cho_solve(jsp.linalg.cho_factor(cov), reshaped_outputs)[0])

Array(True, dtype=bool)

In [36]:
# Check that the all-in one function works accordingly
jnp.allclose(cho_solve(factor, outputs)[0], jsp.linalg.cho_solve((jsp.linalg.cho_factor(cov)[0][0], False), outputs))

Array(True, dtype=bool)

### Shape (B, N, N) + (B, N, N) -> (B, N, N)

In [37]:
key, subkey1, subkey2 = jax.random.split(key, 3)
inputs = jax.random.normal(subkey1, (10, 5, 1))
outputs = jax.random.normal(subkey2, (10, 5, 5))
cov = BatchKernel(kern, batch_size=inputs.shape[0])(inputs)  # Shape (10, 5, 5)
factor = cho_factor(cov)
factor.shape, outputs.shape

((10, 5, 5), (10, 5, 5))

In [38]:
reshaped_outputs = outputs
reshaped_outputs.shape

(10, 5, 5)

In [39]:
jsp.linalg.cho_solve(jsp.linalg.cho_factor(cov), reshaped_outputs).shape

(10, 5, 5)

In [40]:
cho_solve(factor, outputs).shape

(10, 5, 5)

In [41]:
# Check that reshaping doesn't modify results
jnp.allclose(jsp.linalg.cho_solve(jsp.linalg.cho_factor(cov[0]), outputs[0]), jsp.linalg.cho_solve(jsp.linalg.cho_factor(cov), reshaped_outputs)[0])

Array(True, dtype=bool)

In [42]:
# Check that the all-in one function works accordingly
jnp.allclose(cho_solve(factor, outputs)[0], jsp.linalg.cho_solve((jsp.linalg.cho_factor(cov)[0][0], False), outputs[0]))

Array(True, dtype=bool)

### Lifting ambiguity when B == M by adding a dimension: (B, M, M) + (B, M, 1) -> (B, M, 1)

In [43]:
key, subkey1, subkey2 = jax.random.split(key, 3)
inputs = jax.random.normal(subkey1, (10, 5, 1))
outputs = jax.random.normal(subkey2, (10, 5))
cov = BatchKernel(kern, batch_size=inputs.shape[0])(inputs)  # Shape (10, 5, 5)
factor = cho_factor(cov)
factor.shape, outputs.shape

((10, 5, 5), (10, 5))

In [44]:
reshaped_outputs = outputs[:, :, None]
reshaped_outputs.shape

(10, 5, 1)

In [45]:
jsp.linalg.cho_solve(jsp.linalg.cho_factor(cov), reshaped_outputs).shape

(10, 5, 1)

In [46]:
cho_solve(factor, outputs).shape

(10, 5)

In [47]:
# Check that reshaping doesn't modify results
jnp.allclose(jsp.linalg.cho_solve(jsp.linalg.cho_factor(cov[0]), outputs[0]), jsp.linalg.cho_solve(jsp.linalg.cho_factor(cov), reshaped_outputs)[0].squeeze())

Array(True, dtype=bool)

In [48]:
# Check that the all-in one function works accordingly
jnp.allclose(cho_solve(factor, outputs)[0], jsp.linalg.cho_solve((jsp.linalg.cho_factor(cov)[0][0], False), outputs[0]))

Array(True, dtype=bool)

---